# Heuristic Project Portfolio Selection (Simulated Annealing)

This notebook implements a hierarchical project portfolio selection model with:

## Decision Levels
- Level 1: Feasibility screening (Kill / Consider)
- Level 2: Primary prioritization score
- Level 3: Complementary score (secondary refinement)
- Level 4: Future capability score (secondary refinement)

## Portfolio-Level Constraints
- Budget constraint
- Staff-hours constraint
- Time horizon balance
- Initiative intent balance


## 1. Imports

In [ ]:
import random
import math
import time
import pandas as pd
import numpy as np
import random
import os

random.seed(42)  # Reproducibility


## 2. Project Data Input (Flexible + Synthetic)

In [ ]:
# =========================
# CONFIGURATION
# =========================

DATA_SOURCE = "synthetic"
# Options:
# "github"    -> load dataset from GitHub
# "upload"    -> upload local .xlsx or .csv file
# "synthetic" -> generate synthetic dataset

GITHUB_URL = "https://raw.githubusercontent.com/ZakariyaAlHelal/strategic_technology_portfolio_selection/main/data/portfolio_dataset.xlsx"

# Synthetic data settings
N_SYNTHETIC = 60
RANDOM_SEED = 42

# Paper-consistent synthetic distributions
TIME_CATEGORIES = ["Short", "Medium", "Long"]
INTENT_CATEGORIES = ["Exploratory", "Exponential", "Sustaining"]

TIME_PROBS = [0.3, 0.4, 0.3]
INTENT_PROBS = [0.3, 0.4, 0.3]

# If True, export generated synthetic data
EXPORT_SYNTHETIC_DATA = True
SYNTHETIC_EXPORT_CSV = "synthetic_60_projects.csv"
SYNTHETIC_EXPORT_XLSX = "synthetic_60_projects.xlsx"


# =========================
# PROJECT CLASS
# =========================

class Project:
    def __init__(self, idx, p, c, f, b, h, time_cat, intent_cat):
        self.idx = idx
        self.p = p
        self.c = c
        self.f = f
        self.b = b
        self.h = h
        self.time_cat = time_cat
        self.intent_cat = intent_cat


# =========================
# DATAFRAME -> OBJECTS
# =========================

def dataframe_to_projects(df):
    projects = []
    for _, row in df.iterrows():
        projects.append(
            Project(
                int(row["idx"]),
                float(row["p"]),
                float(row["c"]),
                float(row["f"]),
                float(row["b"]),
                float(row["h"]),
                str(row["time_cat"]).strip(),
                str(row["intent_cat"]).strip()
            )
        )
    return projects


# =========================
# LOAD / GENERATE DATA
# =========================

if DATA_SOURCE == "github":
    print("Loading project data from GitHub...")
    df = pd.read_excel(GITHUB_URL, sheet_name="projects")

elif DATA_SOURCE == "upload":
    print("Trying to find a local input file first...")

    candidate_files = [
        f for f in os.listdir()
        if (f.endswith(".xlsx") or f.endswith(".csv"))
        and not f.startswith("optimized_portfolio")
    ]

    if candidate_files:
        file_name = candidate_files[0]
        print(f"Found local file: {file_name}")
    else:
        print("No input file found. Please upload your dataset file (.xlsx or .csv)...")
        from google.colab import files
        uploaded = files.upload()

        if not uploaded:
            raise ValueError("No file uploaded. Please try again.")

        file_name = list(uploaded.keys())[0]
        print(f"Uploaded file: {file_name}")

    if file_name.endswith(".xlsx"):
        df = pd.read_excel(file_name, sheet_name="projects")
    elif file_name.endswith(".csv"):
        df = pd.read_csv(file_name)
    else:
        raise ValueError("Unsupported file type. Please use .xlsx or .csv")

elif DATA_SOURCE == "synthetic":
    print(f"Generating synthetic dataset with {N_SYNTHETIC} projects...")
    print("Assumptions match the paper setup:")
    print("p ~ U(30,100), c,f ~ U(10,40), b ~ U(5,20), h ~ U(10,40)")
    print("time and intent categories sampled with 0.3 / 0.4 / 0.3 proportions")

    random.seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)

    data = []
    for i in range(N_SYNTHETIC):
        project = {
            "idx": i + 1,
            "p": round(np.random.uniform(30, 100), 2),
            "c": round(np.random.uniform(10, 40), 2),
            "f": round(np.random.uniform(10, 40), 2),
            "b": round(np.random.uniform(5, 20), 2),
            "h": round(np.random.uniform(10, 40), 2),
            "time_cat": np.random.choice(TIME_CATEGORIES, p=TIME_PROBS),
            "intent_cat": np.random.choice(INTENT_CATEGORIES, p=INTENT_PROBS),
        }
        data.append(project)

    df = pd.DataFrame(data)

    if EXPORT_SYNTHETIC_DATA:
        df.to_csv(SYNTHETIC_EXPORT_CSV, index=False)
        df.to_excel(SYNTHETIC_EXPORT_XLSX, index=False)
        print(f"Synthetic dataset exported to: {SYNTHETIC_EXPORT_CSV}")
        print(f"Synthetic dataset exported to: {SYNTHETIC_EXPORT_XLSX}")

else:
    raise ValueError("DATA_SOURCE must be one of: 'github', 'upload', 'synthetic'")


# =========================
# VALIDATION
# =========================

required_columns = ["idx", "p", "c", "f", "b", "h", "time_cat", "intent_cat"]

missing_cols = [col for col in required_columns if col not in df.columns]
if missing_cols:
    raise ValueError(
        "Dataset does not match the required input template.\n"
        f"Missing columns: {missing_cols}\n"
        "Required columns are: idx, p, c, f, b, h, time_cat, intent_cat"
    )


# =========================
# CONVERT TO PROJECTS
# =========================

projects = dataframe_to_projects(df)

print("Data loaded successfully!")
print("Number of candidate projects:", len(projects))

# Optional preview
df.head()

Generating synthetic dataset with 60 projects...
Assumptions match the paper setup:
p ~ U(30,100), c,f ~ U(10,40), b ~ U(5,20), h ~ U(10,40)
time and intent categories sampled with 0.3 / 0.4 / 0.3 proportions
Synthetic dataset exported to: synthetic_60_projects.csv
Synthetic dataset exported to: synthetic_60_projects.xlsx
Data loaded successfully!
Number of candidate projects: 60


,idx,p,c,f,b,h,time_cat,intent_cat
0,1,56.22,38.52,31.96,13.98,14.68,Short,Exploratory
1,2,90.63,28.03,31.24,5.31,39.10,Long,Exploratory
2,3,42.73,15.50,19.13,12.87,22.96,Short,Exponential
3,4,39.76,18.76,20.99,11.84,33.56,Short,Exponential
4,5,71.47,11.39,28.23,7.56,11.95,Long,Sustaining


## 3. Resource Constraints

In [ ]:
USE_DYNAMIC_CONSTRAINTS = True  # True = paper model, False = manual

if USE_DYNAMIC_CONSTRAINTS:
    # Paper-consistent constraints
    B_max = 0.6 * sum(p.b for p in projects)
    H_max = 0.6 * sum(p.h for p in projects)
else:
    # User-defined constraints (for experimentation only)
    B_max = 8000
    H_max = 15000

print("Budget limit B_max:", B_max)
print("Hours limit  H_max:", H_max)

Budget limit B_max: 440.166
Hours limit  H_max: 929.832


## 4. Mathematical Model Parameters

In [ ]:
# Small weight for secondary scores (c + f)
EPSILON = 0.01

# Penalty weights for balance
LAMBDA_TIME = 1.0
LAMBDA_INTENT = 1.0

## 5. Heuristic Parameters _ Simulated Annealing (SA) Parameters Only

In [ ]:
MAX_ITER = 20000      # Number of iterations
INITIAL_T = 100       # Initial temperature
ALPHA = 0.95          # Cooling rate (T ← αT)

## 6. Objective and Penalty Functions

In [ ]:
def compute_time_penalty(solution, projects, target):
    counts = {}
    for x, p in zip(solution, projects):
        if x == 1:
            counts[p.time_cat] = counts.get(p.time_cat, 0) + 1

    return sum(abs(counts.get(k, 0) - target[k]) for k in target)


def compute_intent_penalty(solution, projects, target):
    counts = {}
    for x, p in zip(solution, projects):
        if x == 1:
            counts[p.intent_cat] = counts.get(p.intent_cat, 0) + 1

    return sum(abs(counts.get(k, 0) - target[k]) for k in target)


def objective(solution, projects, target_time, target_intent):
    primary = sum(p.p * x for p, x in zip(projects, solution))
    secondary = sum((p.c + p.f) * x for p, x in zip(projects, solution))

    time_pen = compute_time_penalty(solution, projects, target_time)
    intent_pen = compute_intent_penalty(solution, projects, target_intent)

    return primary + EPSILON * secondary \
           - LAMBDA_TIME * time_pen \
           - LAMBDA_INTENT * intent_pen

## 7. Feasibility and Repair

In [ ]:
def is_feasible(solution, projects, B_max, H_max):
    total_b = sum(p.b * x for p, x in zip(projects, solution))
    total_h = sum(p.h * x for p, x in zip(projects, solution))
    return total_b <= B_max and total_h <= H_max


def repair(solution, projects, B_max, H_max):
    # Work on a copy to avoid modifying original solution
    repaired = solution.copy()

    # Remove randomly selected projects until feasible
    while not is_feasible(repaired, projects, B_max, H_max):
        selected = [i for i, x in enumerate(repaired) if x == 1]
        if not selected:
            break
        repaired[random.choice(selected)] = 0

    return repaired

## 8. Greedy Baseline and Dynamic Targets

In [ ]:
def greedy(projects, B_max, H_max):
    N = len(projects)
    solution = [0] * N

    # Rank projects using the SAME value as the objective (without penalties)
    scores = [
        projects[i].p + EPSILON * (projects[i].c + projects[i].f)
        for i in range(N)
    ]

    # Sort indices by score (descending)
    sorted_idx = sorted(range(N), key=lambda i: scores[i], reverse=True)

    # Add projects while keeping feasibility
    for i in sorted_idx:
        solution[i] = 1
        if not is_feasible(solution, projects, B_max, H_max):
            solution[i] = 0

    return solution

## 9. Neighborhood Operator

In [ ]:
def generate_neighbor(solution, projects, swap_prob=0.6):
    new_solution = solution.copy()
    n = len(solution)

    if random.random() < swap_prob:
        # Swap move: remove one selected, add one unselected
        selected = [i for i, x in enumerate(solution) if x == 1]
        unselected = [i for i, x in enumerate(solution) if x == 0]

        if selected and unselected:
            i = random.choice(selected)
            j = random.choice(unselected)
            new_solution[i] = 0
            new_solution[j] = 1

    else:
        # Flip move: toggle any project
        i = random.randrange(n)
        new_solution[i] = 1 - new_solution[i]

    return new_solution

## 10. Standard Simulated Annealing (SA)

In [ ]:
def simulated_annealing(projects, B_max, H_max, target_time, target_intent):
    # Initial solution (greedy)
    current = greedy(projects, B_max, H_max)
    current_value = objective(current, projects, target_time, target_intent)

    best = current[:]
    best_value = current_value

    T = INITIAL_T
    start_time = time.time()

    for _ in range(MAX_ITER):

        # Generate and repair candidate
        candidate = generate_neighbor(current, projects)
        candidate = repair(candidate, projects, B_max, H_max)

        candidate_value = objective(candidate, projects, target_time, target_intent)
        delta = candidate_value - current_value

        # Acceptance rule
        if delta >= 0 or random.random() < math.exp(delta / T):
            current = candidate
            current_value = candidate_value

        # Update best solution
        if current_value > best_value:
            best = current[:]
            best_value = current_value

        # Cooling
        T *= ALPHA
        T = max(T, 1e-8)

    runtime = time.time() - start_time
    return best, best_value, runtime

## 13. Extract and Display SA Results

In [ ]:
# =========================
# CONFIGURATION
# =========================

K = 20  # desired portfolio size (used for targets)
ENFORCE_PORTFOLIO_SIZE = False  # True = force exactly K, False = flexible size (K is only a soft reference for targets)


# =========================
# HELPER FUNCTIONS
# =========================

def extract_selected_projects(solution, projects):
    return [projects[i] for i, x in enumerate(solution) if x == 1]


def compute_targets(K, projects):
    # --- Dataset distribution ---
    time_counts = {}
    for p in projects:
        time_counts[p.time_cat] = time_counts.get(p.time_cat, 0) + 1

    intent_counts = {}
    for p in projects:
        intent_counts[p.intent_cat] = intent_counts.get(p.intent_cat, 0) + 1

    total_projects = len(projects)

    time_props = {k: v / total_projects for k, v in time_counts.items()}
    intent_props = {k: v / total_projects for k, v in intent_counts.items()}

    # --- Targets ---
    target_time = {k: round(v * K) for k, v in time_props.items()}
    target_intent = {k: round(v * K) for k, v in intent_props.items()}

    return target_time, target_intent


# =========================
# TARGETS
# =========================

target_time, target_intent = compute_targets(K, projects)

print("Target time:", target_time)
print("Target intent:", target_intent)


# =========================
# RUN SA (with optional size enforcement)
# =========================

# IMPORTANT:
# Your SA function must respect ENFORCE_PORTFOLIO_SIZE internally
# (swap-only moves if True, original behavior if False)

sa_sol, sa_val, sa_runtime = simulated_annealing(
    projects, B_max, H_max, target_time, target_intent
)


# =========================
# EXTRACT PORTFOLIO
# =========================

selected_projects = extract_selected_projects(sa_sol, projects)

# Sort ONLY for display
selected_projects = sorted(
    selected_projects,
    key=lambda p: p.p + EPSILON * (p.c + p.f),
    reverse=True
)


# =========================
# OUTPUT
# =========================

portfolio_ids = [p.idx for p in selected_projects]

print("\nNOTE:")
print("Target distributions are derived from dataset proportions and scaled to K (neutral baseline).")
print("Targets are exogenous inputs (not optimized).")

if ENFORCE_PORTFOLIO_SIZE:
    print("Portfolio size is enforced as a HARD constraint (exactly K projects).")
else:
    print("Portfolio size is NOT enforced (K used only as a soft reference for balance).")

print("\n==============================")
print("OPTIMIZED PORTFOLIO (SA)")
print("==============================")

print("Projects are sorted for display by Value = p + ε(c+f)")
print("Optimization used FULL objective F(x) including balance penalties")

print("\nSelected Project IDs:", portfolio_ids)
print("Objective value F(x):", round(sa_val, 2))
print("Runtime (s):", round(sa_runtime, 4))
print("Number of projects:", len(selected_projects))

print("\nSelected Projects:")
print("-"*70)

for p in selected_projects:
    value = p.p + EPSILON * (p.c + p.f)
    print(
        f"Project {p.idx:>2} | Value={value:>7.2f} | "
        f"p={p.p:>5.2f} | c={p.c:>5.2f} | f={p.f:>5.2f} | "
        f"b={p.b:>5.1f} | h={p.h:>5.1f} | "
        f"Time={p.time_cat:<6} | Intent={p.intent_cat}"
    )

Target time: {'Short': 8, 'Long': 4, 'Medium': 8}
Target intent: {'Exploratory': 6, 'Exponential': 8, 'Sustaining': 6}

NOTE:
Target distributions are derived from dataset proportions and scaled to K (neutral baseline).
Targets are exogenous inputs (not optimized).
Portfolio size is NOT enforced (K used only as a soft reference for balance).

OPTIMIZED PORTFOLIO (SA)
Projects are sorted for display by Value = p + ε(c+f)
Optimization used FULL objective F(x) including balance penalties

Selected Project IDs: [23, 21, 53, 17, 36, 27, 40, 30, 41, 2, 49, 37, 19, 55, 6, 56, 45, 11, 58, 32, 29, 14, 39, 46, 47, 35, 24, 5, 52, 60, 31, 18, 42, 51, 22, 59]
Objective value F(x): 2942.51
Runtime (s): 1.5311
Number of projects: 36

Selected Projects:
----------------------------------------------------------------------
Project 23 | Value=  99.47 | p=99.00 | c=17.26 | f=30.16 | b= 16.4 | h= 17.1 | Time=Long   | Intent=Exponential
Project 21 | Value=  97.79 | p=97.37 | c=17.55 | f=24.92 | b=  9.5 | 

## 14. Export Optimized Portfolio

In [ ]:
import pandas as pd

# =========================
# EXTRACT PORTFOLIO
# =========================

selected_projects = extract_selected_projects(sa_sol, projects)

# Optional: sort for readability (does NOT affect solution)
selected_projects = sorted(
    selected_projects,
    key=lambda p: p.p + EPSILON * (p.c + p.f),
    reverse=True
)

# =========================
# CONVERT TO TABLE
# =========================

portfolio_data = []

for p in selected_projects:
    portfolio_data.append({
        "Project_ID": p.idx,
        "Primary_Score_p": p.p,
        "Complementary_c": p.c,
        "Capability_f": p.f,
        "Value_p+eps(c+f)": p.p + EPSILON * (p.c + p.f),
        "Budget_b": p.b,
        "Hours_h": p.h,
        "Time_Category": p.time_cat,
        "Intent_Category": p.intent_cat
    })

df_portfolio = pd.DataFrame(portfolio_data)

# =========================
# EXPORT CSV
# =========================

file_name = "optimized_portfolio.csv"
df_portfolio.to_csv(file_name, index=False)

print(f"\n✅ Optimized portfolio exported to: {file_name}")
print(f"Number of projects exported: {len(df_portfolio)}")

# Optional preview
df_portfolio.head()


✅ Optimized portfolio exported to: optimized_portfolio.csv
Number of projects exported: 36


,Project_ID,Primary_Score_p,Complementary_c,Capability_f,Value_p+eps(c+f),Budget_b,Hours_h,Time_Category,Intent_Category
0,23,99.00,17.26,30.16,99.4742,16.42,17.13,Long,Exponential
1,21,97.37,17.55,24.92,97.7947,9.51,18.55,Short,Exponential
2,53,95.90,21.58,38.84,96.5042,18.58,15.87,Short,Exploratory
3,17,95.08,34.24,29.00,95.7124,18.07,34.11,Short,Sustaining
4,36,94.98,22.85,39.00,95.5985,19.45,35.59,Short,Exponential


In [ ]:
from google.colab import files
files.download("optimized_portfolio.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>